# Porównanie Sieci MLP: Float32 vs Fixed-Point (PTQ i QAT)
W tym projekcie badamy wpływ kwantyzacji na rozmiar i dokładność prostej sieci neuronowej trenowanej na zbiorze MNIST.

In [ ]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tabulate import tabulate
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

BATCH_SIZE = 64
FP32_EPOCHS = 5
QAT_EPOCHS = 2
DEVICE = torch.device('cpu')



# Funkcje pomocnicze

In [ ]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tabulate import tabulate
from concurrent.futures import ThreadPoolExecutor


def train_model(model, train_loader, criterion, optimizer, epochs=1, process_name="Train", device='cpu'):
    model.train()
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        print(f"[{process_name}] Zakończono epokę {epoch+1}/{epochs}")

def evaluate_model(model, test_loader, device='cpu'):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(test_loader.dataset)

def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    size = os.path.getsize("temp.p") / 1e3
    os.remove("temp.p")
    return size

def measure_inference_time(model, test_loader, warmup_batches=20, repeats=3, device='cpu'):
    model.eval()
    with torch.no_grad():
        for i, (data, _) in enumerate(test_loader):
            data = data.to(device)
            _ = model(data)
            if i >= warmup_batches:
                break

    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                _ = model(data)
        times.append(time.perf_counter() - start)
    return min(times) * 1000  # ms

def measure_generalization_gap(model, train_loader, test_loader, device='cpu'):
    train_acc = evaluate_model(model, train_loader, device)
    test_acc = evaluate_model(model, test_loader, device)
    return train_acc, test_acc, train_acc - test_acc

def evaluate_with_noise(model, test_loader, noise_std=0.5, device='cpu'):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            noise = torch.randn_like(data) * noise_std
            noisy_data = data + noise
            output = model(noisy_data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    return 100. * correct / len(test_loader.dataset)

def evaluate_test_loss(model, test_loader, criterion, device='cpu'):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item() * data.size(0)
    return total_loss / len(test_loader.dataset)

# Workflow eksperytmentu

Odpowiada za wytrenowanie sieci bazowej F32, PTQ oraz QAT

## F32
Trenujemy standardową sieć bez żadnych modyfikacji.

## PTQ
Kopiujemy wytrenowany model FP32, kalibrujemy go na danych testowych i konwertujemy jego wagi na postać stałoprzecinkową (INT8).

## QAT
Kopiujemy bazowy model, ale poddajemy go dalszemu trenowaniu symulując obcięcie precyzji w trakcie wstecznej propagacji błędu.

In [ ]:
class QuantizationWorkflow:
    def __init__(self, train_loader, test_loader, calibration_loader, criterion, q_engine='x86', device='cpu'):
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.calibration_loader = calibration_loader
        self.criterion = criterion
        self.device = device
        self.q_engine = q_engine

        torch.backends.quantized.engine = self.q_engine
        print(f"[Workflow] Ustawiono silnik kwantyzacji: {self.q_engine}")

    def _run_ptq(self, model):
        """Proces Post-Training Quantization"""
        print("\n[PTQ] Rozpoczęto przygotowanie PTQ...")
        model.eval()
        model.qconfig = torch.ao.quantization.get_default_qconfig(self.q_engine)
        torch.ao.quantization.prepare(model, inplace=True)


        evaluate_model(model, self.calibration_loader, self.device)


        torch.ao.quantization.convert(model, inplace=True)
        print("[PTQ] Zakończono konwersję na Fixed-Point.")
        return model

    def _run_qat(self, model, epochs, lr):
        """Proces Quantization-Aware Training"""
        print(f"\n[QAT] Rozpoczęto proces QAT (Fake Quantization, Epoki: {epochs})...")
        model.train()
        model.qconfig = torch.ao.quantization.get_default_qat_qconfig(self.q_engine)
        torch.ao.quantization.prepare_qat(model, inplace=True)

        optimizer = optim.Adam(model.parameters(), lr=lr)
        train_model(model, self.train_loader, self.criterion, optimizer, epochs=epochs, process_name="QAT", device=self.device)

        model.eval()
        torch.ao.quantization.convert(model, inplace=True)
        print("[QAT] Zakończono dotrenowywanie i konwersję.")
        return model

    def _evaluate_all_metrics(self, model, name, noise_level=0.5):
        """Agreguje wszystkie metryki dla jednego modelu"""
        acc = evaluate_model(model, self.test_loader, self.device)
        size = print_size_of_model(model)
        t_inf = measure_inference_time(model, self.test_loader, device=self.device)
        n_acc = evaluate_with_noise(model, self.test_loader, noise_std=noise_level, device=self.device)
        t_loss = evaluate_test_loss(model, self.test_loader, self.criterion, device=self.device)
        acc_train, acc_test, gap = measure_generalization_gap(model, self.train_loader, self.test_loader, device=self.device)

        return {
            "name": name, "acc": acc, "size": size, "time": t_inf,
            "noise_acc": n_acc, "test_loss": t_loss, "gap": gap,
            "acc_train" : acc_train
        }

    def run(self, base_model, fp32_epochs=5, qat_epochs=2, fp32_lr=0.001, qat_lr=0.0001):
        print(f"\n--- ETAP 1: Wstępny trening bazy FP32 ({fp32_epochs} epok) ---")
        model_base = copy.deepcopy(base_model).to(self.device)
        optimizer_base = optim.Adam(model_base.parameters(), lr=fp32_lr)


        train_model(model_base, self.train_loader, self.criterion, optimizer_base, epochs=fp32_epochs, process_name="FP32-Wstęp", device=self.device)

        print(f"\n--- ETAP 2: Równoległe dotrenowanie FP32 i QAT (+{qat_epochs} epoki) ---")
        model_for_qat = copy.deepcopy(model_base)
        model_fp32_final = copy.deepcopy(model_base) # Ten model dostanie resztę treningu

        def continue_fp32_training(model, epochs, lr):
            print(f"\n[FP32] Dokańczanie treningu bazy (Epoki: {epochs})...")
            optimizer = optim.Adam(model.parameters(), lr=lr)
            train_model(model, self.train_loader, self.criterion, optimizer, epochs=epochs, process_name="FP32-Final", device=self.device)
            return model


        with ThreadPoolExecutor(max_workers=2) as executor:
            fut_qat = executor.submit(self._run_qat, model_for_qat, qat_epochs, qat_lr)
            fut_fp32 = executor.submit(continue_fp32_training, model_fp32_final, qat_epochs, fp32_lr)

            model_qat = fut_qat.result()
            model_fp32_final = fut_fp32.result()

        print("\n--- ETAP 3: PTQ na finalnym modelu FP32 ---")
        model_for_ptq = copy.deepcopy(model_fp32_final)
        model_ptq = self._run_ptq(model_for_ptq)

        print("\n--- ETAP 4: Równoległa ewaluacja i pomiar generalizacji ---")

        with ThreadPoolExecutor(max_workers=3) as executor:
            fut_fp32 = executor.submit(self._evaluate_all_metrics, model_fp32_final, "Float32 (Baza 7 epok)")
            fut_ptq  = executor.submit(self._evaluate_all_metrics, model_ptq, "PTQ (Post-Training 7 epok)")
            fut_qat  = executor.submit(self._evaluate_all_metrics, model_qat, "QAT (Aware Train 7 epok)")

            results = [fut_fp32.result(), fut_ptq.result(), fut_qat.result()]

        self._print_results(results)
        return model_fp32_final, model_ptq, model_qat

    def _print_results(self, results):
        basic_data, adv_data = [], []
        for r in results:
            basic_data.append([r["name"], f"{r['acc']:.2f}%",f"{r['acc_train']:.2f}%", f"{r['size']:.2f} KB", f"{r['time']:.1f} ms"])
            adv_data.append([r["name"], f"{r['noise_acc']:.2f}%", f"{r['test_loss']:.4f}", f"{r['gap']:.4f}"])

        print("\n=== PODSUMOWANIE: WYDAJNOŚĆ I ROZMIAR ===")
        print(tabulate(basic_data, headers=["Model Typ", "Dokładność(Test)","Dokładność(Train)", "Rozmiar", "Czas Inferencji"], tablefmt="grid"))

        print("\n=== PODSUMOWANIE: ZAAWANSOWANA REGULARYZACJA ===")
        print("- 'Acc na szumie': Im wyższa wartość, tym bardziej odporny model.")
        print("- 'Test Loss': Im MNIEJSZA wartość, tym model rzadziej popełnia pewne siebie błędy.")
        print(tabulate(adv_data, headers=["Model Typ", "Acc na szumie", "Test Loss", "GAP Train vs Test"], tablefmt="grid"))

In [ ]:
class QuantizedMLP(nn.Module):
    def __init__(self):
        super(QuantizedMLP, self).__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.quant(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.dequant(x)
        return x

class QuantizedCNN(nn.Module):
    def __init__(self):
        super(QuantizedCNN, self).__init__()
        self.quant = torch.ao.quantization.QuantStub()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()


        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)


        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):

        x = self.quant(x)

        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))

        x = self.flatten(x)

        x = self.relu4(self.fc1(x))
        x = self.fc2(x)


        x = self.dequant(x)
        return x

def get_best_q_engine():
    supported_engines = torch.backends.quantized.supported_engines
    if 'x86' in supported_engines: return 'x86'
    elif 'fbgemm' in supported_engines: return 'fbgemm'
    elif 'qnnpack' in supported_engines: return 'qnnpack'
    return supported_engines[0]

def experiment1():

  transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
  train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
  test_dataset = datasets.MNIST('./data', train=False, transform=transform)

  train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
  test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

  calib_dataset = torch.utils.data.Subset(train_dataset, indices=range(2000))
  calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False)

  base_model = QuantizedMLP()
  criterion = nn.CrossEntropyLoss()

  workflow = QuantizationWorkflow(
    train_loader=train_loader,
    test_loader=test_loader,
    calibration_loader=calib_loader,
    criterion=criterion,
    q_engine=get_best_q_engine(),
    device='cpu'
  )

  fp32, ptq, qat = workflow.run(
    base_model=base_model,
    fp32_epochs=5,
    qat_epochs=2
  )

def experiment2_cifar10():
    print("\n" + "="*50)
    print("ROZPOCZĘCIE EKSPERYMENTU 2: CIFAR-10 + CNN")
    print("="*50)


    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),

        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])


    train_dataset = datasets.CIFAR10('./data_cifar', train=True, download=True, transform=transform_train)
    test_dataset = datasets.CIFAR10('./data_cifar', train=False, download=True, transform=transform_test)


    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


    calib_dataset = torch.utils.data.Subset(train_dataset, indices=range(1500))
    calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False)


    base_model = QuantizedCNN()
    criterion = torch.nn.CrossEntropyLoss()

    workflow = QuantizationWorkflow(
        train_loader=train_loader,
        test_loader=test_loader,
        calibration_loader=calib_loader,
        criterion=criterion,
        q_engine=get_best_q_engine(),
        device='cpu'
    )




    fp32, ptq, qat = workflow.run(
        base_model=base_model,
        fp32_epochs=7,
        qat_epochs=3,
        fp32_lr=0.001,
        qat_lr=0.001
    )
experiment1()
experiment2_cifar10()